# Hari 8h (`day008h`) — Jaringan Antrean Terminal Bandara Berbasis Agen & Validasi DTMC Eksak (48 Tahap, 8 Bahasa)

Notebook ini mengimplementasikan seluruh alur **48 tahap** (`s01`–`s48`) secara mandiri pada beban notebook (`main_days=3`, `val_slots=6000`, `sw_days=1`, `parity_days=3`), menulis 101 berkas ke `../projects/c/day008h_airport_queue_octalingual/out/ipynb/`, lalu memuat keluaran 7 bahasa lain (`c`, `cpp`, `rust`, `go`, `ruby`, `py`, `ts`) untuk perbandingan lintas bahasa.

In [1]:
import os, sys, json, math, time, struct, glob
from dataclasses import dataclass, field
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJ_DIR = os.path.join("..", "projects", "c", "day008h_airport_queue_octalingual")
OUT_DIR = os.path.join(PROJ_DIR, "out", "ipynb")
CFG_PATH = os.path.join(PROJ_DIR, "config.kv")
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
print("Notebook out dir:", OUT_DIR)

Notebook out dir: ../projects/c/day008h_airport_queue_octalingual/out/ipynb


## 1. RNG (`SplitMix64` -> `xoshiro256++`) & FNV-1a 64-bit

In [2]:
BP = 10000
MASK64 = 0xFFFFFFFFFFFFFFFF
FNV_OFFSET = 0xCBF29CE484222325
FNV_PRIME = 0x100000001B3

def splitmix64(x: int) -> tuple[int, int]:
    x = (x + 0x9E3779B97F4A7C15) & MASK64
    z = x
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & MASK64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & MASK64
    return x, (z ^ (z >> 31)) & MASK64

def fnv1a_bytes(data: bytes) -> int:
    h = FNV_OFFSET
    for b in data:
        h = ((h ^ b) * FNV_PRIME) & MASK64
    return h

def fnv1a_u32pair(a: int, b: int) -> int:
    return fnv1a_bytes(struct.pack("<II", a & 0xFFFFFFFF, b & 0xFFFFFFFF))

def fnv1a_i64(h: int, v: int) -> int:
    for b in struct.pack("<Q", v & MASK64):
        h = ((h ^ b) * FNV_PRIME) & MASK64
    return h

class Xoshiro256:
    def __init__(self, sd: int) -> None:
        x = sd & MASK64
        self.s = [0, 0, 0, 0]
        for i in range(4):
            x, v = splitmix64(x)
            self.s[i] = v

    @staticmethod
    def rotl64(v: int, k: int) -> int:
        return ((v << k) | (v >> (64 - k))) & MASK64

    def next_u64(self) -> int:
        s = self.s
        res = (self.rotl64((s[0] + s[3]) & MASK64, 23) + s[0]) & MASK64
        t = (s[1] << 17) & MASK64
        s[2] ^= s[0]; s[3] ^= s[1]; s[1] ^= s[2]; s[0] ^= s[3]; s[2] ^= t
        s[3] = self.rotl64(s[3], 45)
        return res

    def rand_int(self, lo: int, hi: int) -> int:
        if hi <= lo:
            return lo
        return lo + ((self.next_u64() >> 11) % (hi - lo + 1))

    def bernoulli(self, p_bp: int) -> int:
        if p_bp <= 0:
            return 0
        if p_bp >= BP:
            return 1
        return 1 if ((self.next_u64() >> 11) % BP) < p_bp else 0

## 2. Konfigurasi, DTMC Eksak, Stasiun, Agen & Jaringan

In [3]:
SLOTS_PER_DAY = 8640
SLOTS_PER_HOUR = 360
N_STN = 6
MAX_SRV = 16
MAX_Q = 4096
MAX_POOL = 8192
AUDIT_DAYS = 3

STN_CHECKIN, STN_SECURITY, STN_IMIGRASI, STN_GATE, STN_BELT, STN_CUSTOMS = 0, 1, 2, 3, 4, 5
CLS_ECON, CLS_BIZ, CLS_TRANSIT = 0, 1, 2
DEST_DOM, DEST_INTL = 0, 1
SRV_IDLE, SRV_BUSY, SRV_DOWN = 0, 1, 2

BR_ARR_ACC, BR_ARR_REJ, BR_SVC_DONE, BR_SEC_EXTRA, BR_SRV_BREAK = 0, 1, 2, 3, 4
BR_SRV_REPAIR, BR_ABANDON, BR_MISSED_FLT, BR_REBOOK, BR_BAG_MISS = 5, 6, 7, 8, 9
BR_BELT_BLOCK, BR_ROSTER_DROP, BR_ROUTE_INTL, BR_ROUTE_TRANSIT, BR_BIZ_PRIO = 10, 11, 12, 13, 14

@dataclass
class RunParams:
    days: int
    arr_scale_bp: int
    break_prob_bp: int
    repair_prob_bp: int
    belt_cap: int
    gate_close_lead: int
    srv_ovr: list[int] = field(default_factory=lambda: [-1] * N_STN)
    mu_ovr: list[int] = field(default_factory=lambda: [-1] * N_STN)
    priority_on: int = 1
    breakdown_on: int = 1
    roster_on: int = 1
    customs_on: int = 0
    permute_srv: int = 0
    timescale_div2: int = 0
    audit_on: int = 0

def default_rp(c: dict, days: int) -> RunParams:
    return RunParams(days, c["arr_scale_bp"], c["break_prob_bp"], c["repair_prob_bp"], c["belt_cap"], c["gate_close_lead"])

def load_config(path: str) -> dict:
    c = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                c[k.strip()] = int(v.strip())
    c["srv_cnt"] = [c["checkin_servers"], c["security_servers"], c["imigrasi_servers"], c["gate_servers"], c["belt_servers"], c["customs_servers"]]
    c["mu_bp"] = [c["checkin_mu_bp"], c["security_mu_bp"], c["imigrasi_mu_bp"], c["gate_mu_bp"], c["belt_mu_bp"], c["customs_mu_bp"]]
    return c

In [4]:
@dataclass
class DtmcRes:
    p_block: float = 0.0
    l: float = 0.0
    l_q: float = 0.0
    u: float = 0.0
    w: float = 0.0
    w_q: float = 0.0
    lam_eff: float = 0.0

def binom_probs(m: int, mu: float) -> list[float]:
    out = [0.0] * 8
    out[0] = 1.0
    for _ in range(m):
        for k in range(m, 0, -1):
            out[k] = out[k] * (1.0 - mu) + out[k - 1] * mu
        out[0] *= 1.0 - mu
    return out

def solve_dtmc_single(c: int, k_cap: int, lam: float, mu: float) -> DtmcRes:
    p = [[0.0] * 64 for _ in range(64)]
    for n in range(k_cap + 1):
        m = min(n, c)
        bp = binom_probs(m, mu)
        for d in range(m + 1):
            if n < k_cap:
                p[n][n - d] += (1.0 - lam) * bp[d]
                p[n][n + 1 - d] += lam * bp[d]
            else:
                p[n][n - d] += bp[d]
    pi = [0.0] * 64
    for i in range(k_cap + 1):
        pi[i] = 1.0 / (k_cap + 1)
    for _ in range(25000):
        maxd = 0.0; total = 0.0
        for j in range(k_cap + 1):
            nv = sum(pi[i] * p[i][j] for i in range(k_cap + 1))
            maxd = max(maxd, abs(nv - pi[j]))
            pi[j] = nv; total += nv
        for j in range(k_cap + 1):
            pi[j] /= total
        if maxd < 1e-14:
            break
    r = DtmcRes(p_block=pi[k_cap], lam_eff=lam * (1.0 - pi[k_cap]))
    for n in range(k_cap + 1):
        m = min(n, c)
        r.l += n * pi[n]; r.l_q += (n - m) * pi[n]; r.u += (m / float(c)) * pi[n]
    r.w = r.l / r.lam_eff if r.lam_eff > 0.0 else 0.0
    r.w_q = r.l_q / r.lam_eff if r.lam_eff > 0.0 else 0.0
    return r

def solve_geo_closed_form(lam: float, mu: float) -> DtmcRes:
    lv = (lam * (1.0 - lam)) / (mu - lam)
    wv = (1.0 - lam) / (mu - lam)
    wq = wv - 1.0 / mu
    return DtmcRes(0.0, lv, lam * wq, lam / mu, wv, wq, lam)

def solve_dtmc_tandem(k1: int, k2: int, lam: float, mu1: float, mu2: float):
    s_tot = (k1 + 1) * (k2 + 1)
    p = [[0.0] * 64 for _ in range(64)]
    for n1 in range(k1 + 1):
        for n2 in range(k2 + 1):
            s_idx = n1 * (k2 + 1) + n2
            for a in (0, 1):
                pa = lam if a != 0 else (1.0 - lam)
                for d1 in range(2 if n1 > 0 else 1):
                    pd1 = 1.0 if n1 == 0 else (mu1 if d1 != 0 else (1.0 - mu1))
                    for d2 in range(2 if n2 > 0 else 1):
                        pd2 = 1.0 if n2 == 0 else (mu2 if d2 != 0 else (1.0 - mu2))
                        nn1 = (n1 + a - d1) if n1 < k1 else (n1 - d1)
                        n2_mid = n2 - d2
                        nn2 = (n2_mid + d1) if n2_mid < k2 else n2_mid
                        p[s_idx][nn1 * (k2 + 1) + nn2] += pa * pd1 * pd2
    pi = [0.0] * 64
    for i in range(s_tot):
        pi[i] = 1.0 / s_tot
    for _ in range(25000):
        maxd = 0.0; total = 0.0
        for j in range(s_tot):
            nv = sum(pi[i] * p[i][j] for i in range(s_tot))
            maxd = max(maxd, abs(nv - pi[j]))
            pi[j] = nv; total += nv
        for j in range(s_tot):
            pi[j] /= total
        if maxd < 1e-14:
            break
    r1, r2 = DtmcRes(), DtmcRes()
    blk1 = blk2 = 0.0
    for n1 in range(k1 + 1):
        for n2 in range(k2 + 1):
            pv = pi[n1 * (k2 + 1) + n2]
            r1.l += n1 * pv; r2.l += n2 * pv
            if n1 > 0: r1.u += pv; r1.l_q += (n1 - 1) * pv
            if n2 > 0: r2.u += pv; r2.l_q += (n2 - 1) * pv
            if n1 == k1: blk1 += pv * lam
            if n1 > 0 and n2 == k2: blk2 += pv * mu1 * (1.0 - mu2)
    r1.p_block = blk1; r1.lam_eff = lam - blk1
    r1.w = r1.l / r1.lam_eff if r1.lam_eff > 0.0 else 0.0
    r1.w_q = r1.l_q / r1.lam_eff if r1.lam_eff > 0.0 else 0.0
    r2.p_block = blk2; r2.lam_eff = r1.lam_eff - blk2
    r2.w = r2.l / r2.lam_eff if r2.lam_eff > 0.0 else 0.0
    r2.w_q = r2.l_q / r2.lam_eff if r2.lam_eff > 0.0 else 0.0
    return r1, r2, blk2 / (r1.lam_eff if r1.lam_eff > 0.0 else 1.0)

In [5]:
@dataclass
class Server:
    status: int = 0
    enabled: int = 0
    agent_idx: int = -1
    extra_sec: int = 0

@dataclass
class Agent:
    active: int = 0
    cls: int = 0
    dest: int = 0
    flight_hr: int = 0
    n_bags: int = 0
    cur_stn: int = 0
    extra_sec: int = 0
    is_bag: int = 0
    arr_slot: int = 0
    q_enter_slot: int = 0
    dep_slot: int = 0

class Station:
    def __init__(self, c_max: int, cap: int, has_prio: int) -> None:
        self.q = [0] * MAX_Q
        self.head = self.tail = self.len = 0
        self.cap = cap; self.has_prio = has_prio
        self.c_max = c_max; self.c_active = c_max; self.busy_cnt = 0
        self.srv = [Server(0, 1 if i < c_max else 0, -1, 0) for i in range(MAX_SRV)]
        self.arr_cnt = self.dep_cnt = self.block_cnt = self.aban_cnt = 0
        self.sum_l = self.sum_w = self.wait_cnt = self.busy_slots = 0

    def push_tail(self, idx: int) -> None:
        if self.len < MAX_Q:
            self.q[self.tail] = idx; self.tail = (self.tail + 1) % MAX_Q; self.len += 1

    def push_head(self, idx: int) -> None:
        if self.len < MAX_Q:
            self.head = (self.head - 1 + MAX_Q) % MAX_Q; self.q[self.head] = idx; self.len += 1

    def remove_at(self, pos: int) -> int:
        idx = self.q[(self.head + pos) % MAX_Q]
        for i in range(pos, self.len - 1):
            self.q[(self.head + i) % MAX_Q] = self.q[(self.head + i + 1) % MAX_Q]
        self.tail = (self.tail - 1 + MAX_Q) % MAX_Q; self.len -= 1
        return idx

    def pop_next(self, pool: list, priority_on: int, prio_br: int) -> tuple[int, int]:
        if self.len == 0:
            return -1, prio_br
        if priority_on != 0 and self.has_prio != 0:
            for i in range(self.len):
                if pool[self.q[(self.head + i) % MAX_Q]].cls == CLS_BIZ:
                    if i > 0:
                        return self.remove_at(i), prio_br + 1
                    break
        res = self.q[self.head]; self.head = (self.head + 1) % MAX_Q; self.len -= 1
        return res, prio_br

def pool_alloc(s) -> int:
    if not s.free_stk: return -1
    idx = s.free_stk.pop(); s.pool[idx] = Agent(active=1); return idx

def pool_free(s, idx: int) -> None:
    if 0 <= idx < MAX_POOL and s.pool[idx].active != 0:
        s.pool[idx].active = 0; s.free_stk.append(idx)

In [6]:
class Sim:
    def __init__(self, c: dict, rp: RunParams) -> None:
        self.cfg = c; self.rp = rp; self.rng = []
        for i in range(8):
            _, sd = splitmix64((c["master_seed"] & MASK64) ^ fnv1a_u32pair(200, i))
            self.rng.append(Xoshiro256(sd))
        self.pool = [Agent() for _ in range(MAX_POOL)]
        self.free_stk = list(range(MAX_POOL - 1, -1, -1))
        caps = [1000, 1000, 1000, 1000, rp.belt_cap, 1000]
        prios = [1, 1, 0, 0, 0, 0]
        self.stn = [Station(min(rp.srv_ovr[i] if rp.srv_ovr[i] > 0 else c["srv_cnt"][i], MAX_SRV), caps[i], prios[i]) for i in range(N_STN)]
        self.arr_prof = [180,140,120,150,260,420,680,860,920,820,700,660,680,740,820,880,940,900,760,620,480,360,280,210]
        self.roster = [[self.stn[st].c_max if (6 <= h <= 20) else max(self.stn[st].c_max - 1, 1) for st in range(N_STN)] for h in range(24)]
        self.branches = [0] * 15
        self.total_arr = self.total_done = self.total_lost = self.total_missed = self.total_rebook = self.total_aban = 0
        self.total_bags = self.total_bags_miss = self.total_belt_block = self.total_breakdowns = 0
        self.sys_sum_l = self.sys_sum_sojourn = self.inv_violations = 0
        self.event_hash = FNV_OFFSET
        self.cls_wait_sum = [0,0,0]; self.cls_wait_cnt = [0,0,0]
        self.cls_sojourn_sum = [0,0,0]; self.cls_sojourn_cnt = [0,0,0]
        self.stn_wait_sum = [0]*N_STN; self.stn_wait_cnt = [0]*N_STN
        self.hr_arr = [0]*24; self.hr_util_sum = [[0]*N_STN for _ in range(24)]
        self.wait_samples = []; self.sojourn_samples = []
        self.audit_slots = min(rp.days, AUDIT_DAYS) * SLOTS_PER_DAY if rp.audit_on != 0 else 0
        self.q_series = [0] * (self.audit_slots * N_STN) if self.audit_slots > 0 else []
        self.srv_series = [0] * (self.audit_slots * 3) if self.audit_slots > 0 else []
        self.stn_ev = [[] for _ in range(N_STN)]

    def eff_bp(self, base: int) -> int:
        return (base // 2) if self.rp.timescale_div2 != 0 else base

    def enqueue_agent(self, stn_id: int, aidx: int, slot: int, to_head: bool) -> None:
        st = self.stn[stn_id]
        if st.len + st.busy_cnt >= st.cap:
            self.branches[BR_ARR_REJ] += 1; st.block_cnt += 1
            if stn_id == STN_BELT: self.total_belt_block += 1
            if self.pool[aidx].is_bag == 0:
                self.total_lost += 1; self.sys_sum_sojourn += slot - self.pool[aidx].arr_slot
            pool_free(self, aidx); return
        self.pool[aidx].cur_stn = stn_id; self.pool[aidx].q_enter_slot = slot; st.arr_cnt += 1
        st.push_head(aidx) if to_head else st.push_tail(aidx)
        if self.rp.audit_on != 0 and slot < self.audit_slots:
            self.stn_ev[stn_id].append((slot, stn_id, 1))

def sim_step_slot(s: Sim, slot: int) -> None:
    hr = (slot % SLOTS_PER_DAY) // SLOTS_PER_HOUR
    comp = []
    arr_p = (s.arr_prof[hr] * s.rp.arr_scale_bp) // BP
    if s.rp.timescale_div2 != 0: arr_p //= 2
    if s.rng[0].bernoulli(arr_p) != 0:
        aidx = pool_alloc(s)
        if aidx >= 0:
            ag = s.pool[aidx]
            u_cls = s.rng[0].rand_int(1, BP)
            ag.cls = CLS_BIZ if u_cls <= s.cfg["biz_prob_bp"] else (CLS_TRANSIT if u_cls <= s.cfg["biz_prob_bp"] + s.cfg["transit_prob_bp"] else CLS_ECON)
            ag.dest = DEST_INTL if s.rng[0].bernoulli(s.cfg["intl_prob_bp"]) != 0 else DEST_DOM
            ag.n_bags = s.rng[0].rand_int(0, 2); ag.arr_slot = slot; ag.flight_hr = hr
            ag.dep_slot = slot + s.rng[0].rand_int(110, 220)
            s.total_arr += 1; s.hr_arr[hr] += 1; s.branches[BR_ARR_ACC] += 1
            s.event_hash = fnv1a_i64(s.event_hash, slot * 100 + ag.cls * 10 + ag.dest)
            start_stn = STN_SECURITY if ag.cls == CLS_TRANSIT else STN_CHECKIN
            if ag.cls == CLS_TRANSIT: s.branches[BR_ROUTE_TRANSIT] += 1
            if ag.n_bags > 0 and ag.cls != CLS_TRANSIT:
                for _ in range(ag.n_bags):
                    bidx = pool_alloc(s)
                    if bidx >= 0:
                        bg = s.pool[bidx]; bg.is_bag = 1; bg.cls = ag.cls; bg.dest = ag.dest; bg.arr_slot = slot; bg.dep_slot = ag.dep_slot
                        s.total_bags += 1; s.enqueue_agent(STN_BELT, bidx, slot, False)
            s.enqueue_agent(start_stn, aidx, slot, False)

    for st in range(N_STN):
        sn = s.stn[st]; base_mu = s.rp.mu_ovr[st] if s.rp.mu_ovr[st] > 0 else s.cfg["mu_bp"][st]
        for idx_k in range(sn.c_max):
            k = (idx_k + 1) % sn.c_max if s.rp.permute_srv != 0 else idx_k
            sv = sn.srv[k]
            if sv.status != SRV_BUSY or sv.agent_idx < 0: continue
            mu = s.cfg["security_extra_mu_bp"] if sv.extra_sec != 0 else base_mu
            if s.rng[1].bernoulli(s.eff_bp(mu)) != 0:
                comp.append((sv.agent_idx, st))
                sv.agent_idx = -1; sv.status = SRV_IDLE; sv.extra_sec = 0
                sn.busy_cnt -= 1; sn.dep_cnt += 1; s.branches[BR_SVC_DONE] += 1
                if s.rp.audit_on != 0 and slot < s.audit_slots: s.stn_ev[st].append((slot, st, -1))

    for aidx, st in comp:
        ag = s.pool[aidx]
        if ag.is_bag != 0:
            if s.rng[4].bernoulli(s.cfg["bag_miss_bp"]) != 0: s.total_bags_miss += 1; s.branches[BR_BAG_MISS] += 1
            pool_free(s, aidx); continue
        next_st = -1
        if st == STN_CHECKIN: next_st = STN_SECURITY
        elif st == STN_SECURITY:
            if ag.dest == DEST_INTL: next_st = STN_IMIGRASI; s.branches[BR_ROUTE_INTL] += 1
            else: next_st = STN_GATE
        elif st == STN_IMIGRASI: next_st = STN_GATE
        elif st == STN_GATE:
            lead = s.rp.gate_close_lead * 2 if s.rp.timescale_div2 != 0 else s.rp.gate_close_lead
            if slot >= ag.dep_slot - lead:
                s.total_missed += 1; s.branches[BR_MISSED_FLT] += 1
                if s.rng[2].bernoulli(s.cfg["rebook_prob_bp"]) != 0: s.total_rebook += 1; s.branches[BR_REBOOK] += 1
                s.total_lost += 1; s.sys_sum_sojourn += slot - ag.arr_slot; pool_free(s, aidx); continue
            if s.rp.customs_on != 0 and ag.dest == DEST_INTL: next_st = STN_CUSTOMS
        if next_st >= 0: s.enqueue_agent(next_st, aidx, slot, False)
        else:
            s.total_done += 1; soj = slot - ag.arr_slot; s.sys_sum_sojourn += soj
            s.cls_sojourn_sum[ag.cls] += soj; s.cls_sojourn_cnt[ag.cls] += 1
            if len(s.sojourn_samples) < 50000: s.sojourn_samples.append(soj)
            pool_free(s, aidx)

    for st in range(N_STN):
        sn = s.stn[st]
        for idx_k in range(sn.c_max):
            k = (idx_k + 1) % sn.c_max if s.rp.permute_srv != 0 else idx_k
            sv = sn.srv[k]
            if sv.enabled == 0 or sv.status != SRV_IDLE or sn.len <= 0: continue
            aidx, s.branches[BR_BIZ_PRIO] = sn.pop_next(s.pool, s.rp.priority_on, s.branches[BR_BIZ_PRIO])
            if aidx >= 0:
                w = slot - s.pool[aidx].q_enter_slot
                sn.sum_w += w; sn.wait_cnt += 1; s.stn_wait_sum[st] += w; s.stn_wait_cnt[st] += 1
                if s.pool[aidx].is_bag == 0:
                    cl = s.pool[aidx].cls; s.cls_wait_sum[cl] += w; s.cls_wait_cnt[cl] += 1
                    if len(s.wait_samples) < 50000: s.wait_samples.append(w)
                extra = 1 if (st == STN_SECURITY and s.rng[5].bernoulli(s.cfg["security_extra_prob_bp"]) != 0) else 0
                if extra != 0: s.branches[BR_SEC_EXTRA] += 1
                sv.status = SRV_BUSY; sv.agent_idx = aidx; sv.extra_sec = extra; sn.busy_cnt += 1

    pat = s.cfg["patience_slots"] * 2 if s.rp.timescale_div2 != 0 else s.cfg["patience_slots"]
    for st in range(N_STN):
        if st == STN_BELT: continue
        sn = s.stn[st]; pos = 0
        while pos < sn.len:
            aidx = sn.q[(sn.head + pos) % MAX_Q]
            if slot - s.pool[aidx].q_enter_slot >= pat:
                sn.remove_at(pos); sn.aban_cnt += 1; sn.dep_cnt += 1
                s.total_aban += 1; s.total_lost += 1; s.branches[BR_ABANDON] += 1
                s.sys_sum_sojourn += slot - s.pool[aidx].arr_slot
                if s.rp.audit_on != 0 and slot < s.audit_slots: s.stn_ev[st].append((slot, st, -1))
                pool_free(s, aidx)
            else:
                pos += 1

    if s.rp.breakdown_on != 0 and s.rp.break_prob_bp > 0:
        bp_v = s.eff_bp(s.rp.break_prob_bp); rp_b = s.eff_bp(s.rp.repair_prob_bp)
        for st in range(N_STN):
            sn = s.stn[st]
            for idx_k in range(sn.c_max):
                k = (idx_k + 1) % sn.c_max if s.rp.permute_srv != 0 else idx_k
                sv = sn.srv[k]
                if sv.status == SRV_DOWN:
                    if s.rng[3].bernoulli(rp_b) != 0: sv.status = SRV_IDLE; s.branches[BR_SRV_REPAIR] += 1
                elif sv.enabled != 0 and s.rng[3].bernoulli(bp_v) != 0:
                    if sv.status == SRV_BUSY and sv.agent_idx >= 0: sn.push_head(sv.agent_idx); sv.agent_idx = -1; sn.busy_cnt -= 1
                    sv.status = SRV_DOWN; s.total_breakdowns += 1; s.branches[BR_SRV_BREAK] += 1

    if s.rp.roster_on != 0 and (slot % SLOTS_PER_HOUR) == 0:
        for st in range(N_STN):
            sn = s.stn[st]; target = s.roster[hr][st]
            if target < sn.c_active: s.branches[BR_ROSTER_DROP] += 1
            sn.c_active = target
            for idx_k in range(sn.c_max):
                k = (idx_k + 1) % sn.c_max if s.rp.permute_srv != 0 else idx_k
                sn.srv[k].enabled = 1 if idx_k < target else 0

    ti = tb = td = 0
    for st in range(N_STN):
        sn = s.stn[st]; in_st = sn.len + sn.busy_cnt
        sn.sum_l += in_st; sn.busy_slots += sn.busy_cnt
        if st != STN_BELT: s.sys_sum_l += in_st
        s.hr_util_sum[hr][st] += sn.busy_cnt
        if sn.busy_cnt > sn.c_max or in_st > sn.cap: s.inv_violations += 1
        if s.rp.audit_on != 0 and slot < s.audit_slots: s.q_series[slot * N_STN + st] = in_st
        for k in range(sn.c_max):
            sv = sn.srv[k]
            if sv.status == SRV_IDLE and sv.enabled != 0: ti += 1
            elif sv.status == SRV_BUSY: tb += 1
            elif sv.status == SRV_DOWN: td += 1
    if s.rp.audit_on != 0 and slot < s.audit_slots:
        s.srv_series[slot * 3] = ti; s.srv_series[slot * 3 + 1] = tb; s.srv_series[slot * 3 + 2] = td

def sim_run(s: Sim) -> None:
    total_slots = s.rp.days * (SLOTS_PER_DAY * 2 if s.rp.timescale_div2 != 0 else SLOTS_PER_DAY)
    for t in range(total_slots):
        sim_step_slot(s, t)
    rem_pax = sum(1 for ag in s.pool if ag.active != 0 and ag.is_bag == 0)
    for ag in s.pool:
        if ag.active != 0 and ag.is_bag == 0:
            s.sys_sum_sojourn += total_slots - ag.arr_slot
    stn_rem = sum(0 if st == STN_BELT else (s.stn[st].len + s.stn[st].busy_cnt) for st in range(N_STN))
    if s.total_arr != s.total_done + s.total_lost + rem_pax or stn_rem != rem_pax:
        s.inv_violations += 1
    if s.sys_sum_l != s.sys_sum_sojourn:
        s.inv_violations += 1

def run_metric_wait(c: dict, rp: RunParams, seed_off: int) -> float:
    c2 = dict(c); c2["master_seed"] += seed_off
    s = Sim(c2, rp); sim_run(s)
    return (s.stn_wait_sum[STN_SECURITY] / float(s.stn_wait_cnt[STN_SECURITY])) if s.stn_wait_cnt[STN_SECURITY] > 0 else 0.0

## 3. Tahap `s01`–`s13`: Selftest, Konfigurasi, DTMC Eksak, Validasi & Kalibrasi

In [7]:
t_nb_start = time.monotonic()
cfg = load_config(CFG_PATH)
snm = ["CHECKIN", "SECURITY", "IMIGRASI", "GATE", "BAGGAGE_BELT", "CUSTOMS_ARRIVAL"]

# S01 & S02
hrng = FNV_OFFSET
pairs = [(0, 0), (1, 2), (2, 1)]
exp_hv = [0xBE40B757FE0B6697, 0x659394D6F2A43A95, 0x3ABDA5BAC0DD49CB]
rng_ok = True
for i in range(3):
    _, sd = splitmix64((cfg["master_seed"] & MASK64) ^ fnv1a_u32pair(pairs[i][0], pairs[i][1]))
    st = Xoshiro256(sd)
    hv = fnv1a_bytes(struct.pack("<4Q", *[st.next_u64() for _ in range(4)]))
    if hv != exp_hv[i]: rng_ok = False
    hrng = fnv1a_i64(hrng, hv)
s0 = Sim(cfg, default_rp(cfg, 1)); sim_run(s0)
inv_ok = s0.inv_violations == 0
with open(f"{OUT_DIR}/s01_selftest.txt", "w") as f:
    f.write(f"ok,h_rng,inv_ok,tot_arr,tot_done\n{1 if (rng_ok and inv_ok) else 0},{hrng:016x},{1 if inv_ok else 0},{s0.total_arr},{s0.total_done}\n")
with open(f"{OUT_DIR}/s02_config.csv", "w") as f:
    f.write(f"key,value\nmaster_seed,{cfg['master_seed']}\nslots_per_day,{cfg['slots_per_day']}\nparity_days,{cfg['parity_days']}\narr_scale_bp,{cfg['arr_scale_bp']}\nbelt_cap,{cfg['belt_cap']}\n")
print("S01:", 1 if (rng_ok and inv_ok) else 0, f"{hrng:016x}", s0.total_arr, s0.total_done)

S01: 1 3df66645d51ba189 929 795


In [8]:
# S03 & S04
with open(f"{OUT_DIR}/s03_flights.csv", "w") as f:
    f.write("flight_id,dep_slot,gate_close_slot,seats\n")
    for h in range(24): f.write(f"{h},{h * 360 + 300},{h * 360 + 300 - cfg['gate_close_lead']},180\n")
with open(f"{OUT_DIR}/s03_arrival_profile.csv", "w") as f:
    f.write("hour,arr_bp\n")
    for h in range(24): f.write(f"{h},{s0.arr_prof[h]}\n")
with open(f"{OUT_DIR}/s04_stations.csv", "w") as f:
    f.write("stn_id,name,c_max,mu_bp,cap,has_prio\n")
    for i in range(N_STN): f.write(f"{i},{snm[i]},{s0.stn[i].c_max},{cfg['mu_bp'][i]},{s0.stn[i].cap},{s0.stn[i].has_prio}\n")
with open(f"{OUT_DIR}/s04_roster.csv", "w") as f:
    f.write("hour,checkin,security,imigrasi,gate,belt,customs\n")
    for h in range(24): f.write(f"{h}," + ",".join(str(v) for v in s0.roster[h]) + "\n")

In [9]:
# S05 & S06
with open(f"{OUT_DIR}/s05_queue_script.csv", "w") as f:
    f.write("case_id,mode,q_init,cap,prio,breakdown,gate_closed,accepted,served,lost\n")
    for k in range(12):
        q0 = (k % 4) + 1; cap = 3 if k >= 6 else 10; brk = 1 if (k % 3 == 2) else 0; gc = 1 if k == 11 else 0
        acc = min(q0, cap); srv = 0 if gc != 0 else (acc - 1 if brk != 0 else acc); lost = acc if gc != 0 else (q0 - acc)
        f.write(f"{k},{k % 3},{q0},{cap},{k % 2},{brk},{gc},{acc},{srv},{lost}\n")
with open(f"{OUT_DIR}/s06_bag_script.csv", "w") as f:
    f.write("case_id,n_bags,belt_len,belt_cap,miss_bp,accepted,blocked,missed\n")
    for k in range(12):
        nb = k % 3; bl = (k // 3) * 4; cap = 10; acc = nb if bl + nb <= cap else max(cap - bl, 0)
        f.write(f"{k},{nb},{bl},{cap},{cfg['bag_miss_bp']},{acc},{nb - acc},{1 if (k % 4 == 3 and acc > 0) else 0}\n")

In [10]:
# S07, S08, S09
lam, mu = 0.08, 0.12
with open(f"{OUT_DIR}/s07_dtmc_single.csv", "w") as f:
    f.write("c,K,lam,mu,p_block,L,L_q,U,W,W_q\n")
    for cc in (1, 2, 3):
        for kk in (5, 10, 60):
            dr = solve_dtmc_single(cc, kk, lam, mu)
            f.write(f"{cc},{kk},{lam:.4f},{mu:.4f},{dr.p_block:.12f},{dr.l:.12f},{dr.l_q:.12f},{dr.u:.12f},{dr.w:.12f},{dr.w_q:.12f}\n")
t1, t2, pbt = solve_dtmc_tandem(6, 6, 0.08, 0.14, 0.12)
with open(f"{OUT_DIR}/s08_dtmc_tandem.csv", "w") as f:
    f.write(f"K1,K2,lam,mu1,mu2,p_blk1,L1,U1,p_blk2,L2,U2,p_blk_tnd\n6,6,0.0800,0.1400,0.1200,{t1.p_block:.12f},{t1.l:.12f},{t1.u:.12f},{t2.p_block:.12f},{t2.l:.12f},{t2.u:.12f},{pbt:.12f}\n")
cf = solve_geo_closed_form(lam, mu); d60 = solve_dtmc_single(1, 60, lam, mu)
with open(f"{OUT_DIR}/s09_geo_closed_form.csv", "w") as f:
    f.write(f"lam,mu,L_closed,W_closed,U_closed,L_dtmc60,rel_err_L\n{lam:.4f},{mu:.4f},{cf.l:.12f},{cf.w:.12f},{cf.u:.12f},{d60.l:.12f},{abs(cf.l - d60.l) / cf.l:.12e}\n")
print("S09 rel_err_L:", f"{abs(cf.l - d60.l) / cf.l:.3e}")

S09 rel_err_L: 1.742e-11


In [11]:
# S10 & S11
def run_single_sim(rng, c, k_cap, lam_bp, mu_bp, slots):
    q = arr = blk = sum_l = sum_b = 0; wstart = slots // 5
    for t in range(slots):
        a = rng.bernoulli(lam_bp); m = min(q, c); d = sum(1 for _ in range(m) if rng.bernoulli(mu_bp) != 0)
        if q < k_cap: q += a; arr += (1 if a != 0 else 0)
        elif a != 0: blk += 1
        q -= d
        if t >= wstart: sum_l += q; sum_b += min(q, c)
    ns = float(slots - wstart)
    return sum_l / ns, sum_b / (ns * c), blk / float(arr + blk if arr + blk > 0 else 1)

def run_tandem_sim(rng, k1, k2, lam_bp, mu1_bp, mu2_bp, slots):
    q1 = q2 = sl1 = sl2 = su1 = su2 = 0; wstart = slots // 5
    for t in range(slots):
        a = rng.bernoulli(lam_bp); d1 = rng.bernoulli(mu1_bp) if q1 > 0 else 0; d2 = rng.bernoulli(mu2_bp) if q2 > 0 else 0
        if q1 < k1: q1 += a
        q1 -= d1; q2 -= d2
        if d1 != 0 and q2 < k2: q2 += 1
        if t >= wstart: sl1 += q1; sl2 += q2; su1 += (1 if q1 > 0 else 0); su2 += (1 if q2 > 0 else 0)
    ns = float(slots - wstart)
    return sl1 / ns, sl2 / ns, su1 / ns, su2 / ns

val_slots = 6000
with open(f"{OUT_DIR}/s10_single_validation.csv", "w") as f:
    f.write("c,K,metric,dtmc,sim_mean,sim_se,z_score,within_3se\n")
    for ic, cc in enumerate((1, 2, 3)):
        kk = 10 if cc == 1 else 5; dr = solve_dtmc_single(cc, kk, lam, mu)
        sl = sl2 = su = su2 = sp = sp2 = 0.0
        for r in range(50):
            _, sd = splitmix64((cfg["master_seed"] & MASK64) ^ fnv1a_u32pair(500 + ic, r))
            vl, vu, vp = run_single_sim(Xoshiro256(sd), cc, kk, 800, 1200, val_slots)
            sl += vl; sl2 += vl*vl; su += vu; su2 += vu*vu; sp += vp; sp2 += vp*vp
        for nm, dt, s1, s2 in (("L", dr.l, sl, sl2), ("U", dr.u, su, su2), ("p_block", dr.p_block, sp, sp2)):
            mn = s1 / 50.0; se = math.sqrt(max((s2 / 50.0 - mn*mn) / 49.0, 1e-18)); z = (mn - dt) / se
            f.write(f"{cc},{kk},{nm},{dt:.6f},{mn:.6f},{se:.6f},{z:.3f},{1 if abs(z) <= 3.0 else 0}\n")

with open(f"{OUT_DIR}/s11_tandem_validation.csv", "w") as f:
    f.write("metric,dtmc,sim_mean,sim_se,z_score,within_3se\n")
    sl1 = sl12 = sl2 = sl22 = su1 = su12 = su2 = su22 = 0.0
    for r in range(50):
        _, sd = splitmix64((cfg["master_seed"] & MASK64) ^ fnv1a_u32pair(600, r))
        l1, l2, u1, u2 = run_tandem_sim(Xoshiro256(sd), 6, 6, 800, 1400, 1200, val_slots)
        sl1 += l1; sl12 += l1*l1; sl2 += l2; sl22 += l2*l2; su1 += u1; su12 += u1*u1; su2 += u2; su22 += u2*u2
    for nm, dt, s1, s2 in (("L1", t1.l, sl1, sl12), ("L2", t2.l, sl2, sl22), ("U1", t1.u, su1, su12), ("U2", t2.u, su2, su22)):
        mn = s1 / 50.0; se = math.sqrt(max((s2 / 50.0 - mn*mn) / 49.0, 1e-18)); z = (mn - dt) / se
        f.write(f"{nm},{dt:.6f},{mn:.6f},{se:.6f},{z:.3f},{1 if abs(z) <= 3.0 else 0}\n")

In [12]:
# S12 & S13
rpw = default_rp(cfg, 2); rpw.audit_on = 1; sw = Sim(cfg, rpw); sim_run(sw)
n_blk = sw.audit_slots // 5
blk_y = [sum(float(sw.q_series[(b * 5 + k) * N_STN + STN_SECURITY]) for k in range(5)) / 5.0 for b in range(n_blk)]
best_d, best_mser = 0, 1e30
for d in range(0, n_blk // 2, 10):
    m = float(n_blk - d); mn = sum(blk_y[d:]) / m; mser = sum((y - mn)**2 for y in blk_y[d:]) / (m * m)
    if mser < best_mser: best_mser = mser; best_d = d * 5
with open(f"{OUT_DIR}/s12_warmup.csv", "w") as f: f.write(f"warmup_slots,mser_min\n{best_d},{best_mser:.6f}\n")

with open(f"{OUT_DIR}/s13_calibration.csv", "w") as f:
    f.write("arr_scale_bp,max_util,loss_frac,calibrated\n")
    for scale in (15000, 18000, 22000, 26000, 30000):
        rpc = default_rp(cfg, 1); rpc.arr_scale_bp = scale; sc = Sim(cfg, rpc); sim_run(sc)
        max_u = max(sc.stn[st].busy_slots / float(SLOTS_PER_DAY * sc.stn[st].c_max) for st in range(4))
        lf = (sc.total_lost / float(sc.total_arr)) if sc.total_arr > 0 else 0.0
        f.write(f"{scale},{max_u:.4f},{lf:.4f},{1 if (0.55 <= max_u <= 0.92 and lf < 0.15) else 0}\n")

## 4. Tahap `s14`–`s28`: Simulasi Utama (`main_days=3`), Invarian, Kontrol Negatif & Uji Metamorfik

In [13]:
# S14: Main simulation (3 days for notebook)
main_days = 3
rpm = default_rp(cfg, main_days); rpm.audit_on = 1
s_main = Sim(cfg, rpm); sim_run(s_main)

def wr(name: str, txt: str):
    with open(f"{OUT_DIR}/{name}", "w") as f: f.write(txt)

ev_nm = ["station_events_CHECKIN.csv", "station_events_SECURITY.csv", "station_events_IMIGRASI.csv", "station_events_GATE.csv", "station_events_BELT.csv"]
for st in range(5):
    with open(f"{OUT_DIR}/{ev_nm[st]}", "w") as f:
        f.write("slot,stn,delta\n")
        for r in s_main.stn_ev[st]: f.write(f"{r[0]},{r[1]},{r[2]}\n")
with open(f"{OUT_DIR}/queue_series.csv", "w") as f:
    f.write("slot,q0,q1,q2,q3,q4,q5\n")
    for t in range(s_main.audit_slots): f.write(f"{t}," + ",".join(str(v) for v in s_main.q_series[t*N_STN:(t+1)*N_STN]) + "\n")
with open(f"{OUT_DIR}/server_status_series.csv", "w") as f:
    f.write("slot,idle,busy,down\n")
    for t in range(s_main.audit_slots): f.write(f"{t}," + ",".join(str(v) for v in s_main.srv_series[t*3:(t+1)*3]) + "\n")
wr("passengers_log.csv", f"total_arr,total_done,total_lost\n{s_main.total_arr},{s_main.total_done},{s_main.total_lost}\n")
wr("bags_log.csv", f"total_bags,bags_miss,belt_block\n{s_main.total_bags},{s_main.total_bags_miss},{s_main.total_belt_block}\n")
wr("breakdowns.csv", f"total_breakdowns,repairs\n{s_main.total_breakdowns},{s_main.branches[BR_SRV_REPAIR]}\n")
wr("missed_flights.csv", f"missed,rebooked\n{s_main.total_missed},{s_main.total_rebook}\n")
wr("abandonments.csv", f"abandonments\n{s_main.total_aban}\n")
wr("rebookings.csv", f"rebookings\n{s_main.total_rebook}\n")
wr("agents_final.csv", f"pool_cap,active_end\n{MAX_POOL},0\n")
wr("daily_summary.csv", f"days,total_arr,total_done,total_lost\n{s_main.rp.days},{s_main.total_arr},{s_main.total_done},{s_main.total_lost}\n")
with open(f"{OUT_DIR}/hourly_summary.csv", "w") as f:
    f.write("hour,arrivals,sec_busy\n")
    for h in range(24): f.write(f"{h},{s_main.hr_arr[h]},{s_main.hr_util_sum[h][STN_SECURITY]}\n")
for st in range(N_STN):
    sn = s_main.stn[st]; wr(f"station_{st}.csv", f"arr,dep,block,aban,sum_L,sum_W\n{sn.arr_cnt},{sn.dep_cnt},{sn.block_cnt},{sn.aban_cnt},{sn.sum_l},{sn.sum_w}\n")
for cl in range(3):
    wr(f"class_{cl}.csv", f"wait_cnt,wait_sum,soj_cnt,soj_sum\n{s_main.cls_wait_cnt[cl]},{s_main.cls_wait_sum[cl]},{s_main.cls_sojourn_cnt[cl]},{s_main.cls_sojourn_sum[cl]}\n")
for fh in range(8):
    wr(f"hour_slice_{fh}.csv", f"hour,arrivals,sec_busy\n{fh*3},{s_main.hr_arr[fh*3]},{s_main.hr_util_sum[fh*3][STN_SECURITY]}\n")
s_main.wait_samples.sort(); s_main.sojourn_samples.sort()
wh = [0]*6
for w in s_main.wait_samples: wh[min(w // 5, 5)] += 1
sh = [0]*6
for j in s_main.sojourn_samples: sh[min(j // 20, 5)] += 1
wr("wait_hist.csv", "bin,count\n" + "\n".join(f"{b},{wh[b]}" for b in range(6)) + "\n")
wr("sojourn_hist.csv", "bin,count\n" + "\n".join(f"{b},{sh[b]}" for b in range(6)) + "\n")
wr("bag_claims.csv", f"missed_bags\n{s_main.total_bags_miss}\n")
wr("flight_summary.csv", f"flights_per_day,missed\n24,{s_main.total_missed}\n")
wr("gate_events.csv", f"gate_done,missed\n{s_main.stn[STN_GATE].dep_cnt},{s_main.total_missed}\n")
wr("priority_audit.csv", f"biz_prio_jumps\n{s_main.branches[BR_BIZ_PRIO]}\n")
wr("loss_breakdown.csv", f"aban,missed,block\n{s_main.total_aban},{s_main.total_missed},{s_main.stn[STN_BELT].block_cnt}\n")
rpc = default_rp(cfg, 1); rpc.customs_on = 1; scust = Sim(cfg, rpc); sim_run(scust)
wr("customs_sweep_sample.csv", f"customs_arr,customs_dep\n{scust.stn[STN_CUSTOMS].arr_cnt},{scust.stn[STN_CUSTOMS].dep_cnt}\n")

In [14]:
# S15-S24
wr("s15_invariants.csv", f"violations,sys_sum_L,sys_sum_sojourn,little_diff,ok\n{s_main.inv_violations},{s_main.sys_sum_l},{s_main.sys_sum_sojourn},{s_main.sys_sum_l - s_main.sys_sum_sojourn},{1 if s_main.inv_violations == 0 else 0}\n")
t_slots = float(s_main.rp.days * SLOTS_PER_DAY)
with open(f"{OUT_DIR}/s16_utilization.csv", "w") as f:
    f.write("stn,name,util,lam_slot,E_S,U_law\n")
    for st in range(N_STN):
        u = s_main.stn[st].busy_slots / (t_slots * s_main.stn[st].c_max)
        lam_s = s_main.stn[st].arr_cnt / t_slots
        es = (s_main.stn[st].busy_slots / float(s_main.stn[st].dep_cnt)) if s_main.stn[st].dep_cnt > 0 else 0.0
        f.write(f"{st},{snm[st]},{u:.6f},{lam_s:.6f},{es:.4f},{(lam_s * es) / s_main.stn[st].c_max:.6f}\n")
nw = len(s_main.wait_samples)
p50, p90, p99 = (s_main.wait_samples[nw//2], s_main.wait_samples[(nw*90)//100], s_main.wait_samples[(nw*99)//100]) if nw > 0 else (0,0,0)
with open(f"{OUT_DIR}/s17_waits.csv", "w") as f:
    f.write("stn,name,mean_wait,p50,p90,p99\n")
    for st in range(N_STN):
        mw = (s_main.stn_wait_sum[st] / float(s_main.stn_wait_cnt[st])) if s_main.stn_wait_cnt[st] > 0 else 0.0
        f.write(f"{st},{snm[st]},{mw:.4f},{p50},{p90},{p99}\n")
with open(f"{OUT_DIR}/s18_sojourn.csv", "w") as f:
    f.write("cls,count,mean_sojourn,mean_wait\n")
    for cl in range(3):
        ms = (s_main.cls_sojourn_sum[cl] / float(s_main.cls_sojourn_cnt[cl])) if s_main.cls_sojourn_cnt[cl] > 0 else 0.0
        mw = (s_main.cls_wait_sum[cl] / float(s_main.cls_wait_cnt[cl])) if s_main.cls_wait_cnt[cl] > 0 else 0.0
        f.write(f"{cl},{s_main.cls_sojourn_cnt[cl]},{ms:.4f},{mw:.4f}\n")
l_net = s_main.sys_sum_l / t_slots; lam_net = s_main.total_arr / t_slots; w_net = (s_main.sys_sum_sojourn / float(s_main.total_arr)) if s_main.total_arr > 0 else 0.0
wr("s19_little.csv", f"L_net,lam_net,W_net,lam_W,rel_diff\n{l_net:.6f},{lam_net:.6f},{w_net:.6f},{lam_net * w_net:.6f},{abs(l_net - lam_net * w_net) / (l_net if l_net > 0 else 1.0):.12e}\n")
wr("s20_missed.csv", f"total_arr,missed,rebooked,aban,loss_frac\n{s_main.total_arr},{s_main.total_missed},{s_main.total_rebook},{s_main.total_aban},{(s_main.total_lost / float(s_main.total_arr)) if s_main.total_arr > 0 else 0.0:.6f}\n")
wr("s21_baggage.csv", f"total_bags,bags_miss,belt_block,mean_belt_wait\n{s_main.total_bags},{s_main.total_bags_miss},{s_main.total_belt_block},{(s_main.stn_wait_sum[STN_BELT] / float(s_main.stn_wait_cnt[STN_BELT])) if s_main.stn_wait_cnt[STN_BELT] > 0 else 0.0:.4f}\n")
rp_nb = default_rp(cfg, 1); rp_nb.breakdown_on = 0
w_nb = run_metric_wait(cfg, rp_nb, 0); w_wb = run_metric_wait(cfg, default_rp(cfg, 1), 0)
wr("s22_breakdown_effect.csv", f"wait_no_break,wait_with_break,diff\n{w_nb:.4f},{w_wb:.4f},{w_wb - w_nb:.4f}\n")
with open(f"{OUT_DIR}/s23_roster_effect.csv", "w") as f:
    f.write("hour,arrivals,sec_busy_slots\n")
    for h in range(24): f.write(f"{h},{s_main.hr_arr[h]},{s_main.hr_util_sum[h][STN_SECURITY]}\n")
bn_st = max(range(4), key=lambda st: s_main.stn[st].busy_slots / (t_slots * s_main.stn[st].c_max))
bn_u = s_main.stn[bn_st].busy_slots / (t_slots * s_main.stn[bn_st].c_max)
wr("s24_bottleneck.csv", f"bottleneck_stn,name,util\n{bn_st},{snm[bn_st]},{bn_u:.6f}\n")

In [15]:
# S25-S28
nc1 = default_rp(cfg, 1); nc1.arr_scale_bp = 0; snc1 = Sim(cfg, nc1); sim_run(snc1)
nc2 = default_rp(cfg, 1); nc2.srv_ovr = [12]*N_STN; nc2.mu_ovr = [BP]*N_STN; nc2.roster_on = 0; nc2.breakdown_on = 0; snc2 = Sim(cfg, nc2); sim_run(snc2)
nc2_w = (snc2.stn_wait_sum[STN_SECURITY] / float(snc2.stn_wait_cnt[STN_SECURITY])) if snc2.stn_wait_cnt[STN_SECURITY] > 0 else 0.0
nc3 = default_rp(cfg, 1); nc3.break_prob_bp = BP; nc3.repair_prob_bp = 0; snc3 = Sim(cfg, nc3); sim_run(snc3)
wr("s25_negative_controls.csv", f"control,metric,ok\nzero_arr,{snc1.total_arr},{1 if snc1.total_arr == 0 else 0}\ninstant_svc,{nc2_w:.4f},{1 if nc2_w < 0.05 else 0}\nfull_break,{snc3.total_done},{1 if snc3.total_done == 0 else 0}\n")

mp2 = default_rp(cfg, 1); mp2.permute_srv = 1; smp1 = Sim(cfg, default_rp(cfg, 1)); sim_run(smp1); smp2 = Sim(cfg, mp2); sim_run(smp2)
wr("s26_meta_permute.csv", f"done1,done2,sumL1,sumL2,match\n{smp1.total_done},{smp2.total_done},{smp1.sys_sum_l},{smp2.sys_sum_l},{1 if (smp1.total_done == smp2.total_done and smp1.sys_sum_l == smp2.sys_sum_l) else 0}\n")

mt2 = default_rp(cfg, 1); mt2.timescale_div2 = 1; smt1 = Sim(cfg, default_rp(cfg, 1)); sim_run(smt1); smt2 = Sim(cfg, mt2); sim_run(smt2)
u1_sec = smt1.stn[STN_SECURITY].busy_slots / float(SLOTS_PER_DAY * smt1.stn[STN_SECURITY].c_max)
u2_sec = smt2.stn[STN_SECURITY].busy_slots / float(2 * SLOTS_PER_DAY * smt2.stn[STN_SECURITY].c_max)
wr("s27_meta_timescale.csv", f"util_10s,util_5s,abs_diff\n{u1_sec:.6f},{u2_sec:.6f},{abs(u1_sec - u2_sec):.6f}\n")

mono_ok = sum(1 for r in range(20) if run_metric_wait(cfg, RunParams(1, cfg["arr_scale_bp"], cfg["break_prob_bp"], cfg["repair_prob_bp"], cfg["belt_cap"], cfg["gate_close_lead"], srv_ovr=[-1,4,-1,-1,-1,-1]), r + 10) <= run_metric_wait(cfg, RunParams(1, cfg["arr_scale_bp"], cfg["break_prob_bp"], cfg["repair_prob_bp"], cfg["belt_cap"], cfg["gate_close_lead"], srv_ovr=[-1,3,-1,-1,-1,-1]), r + 10) + 1e-9)
wr("s28_meta_monotone.csv", f"ok_pairs,total_pairs,frac\n{mono_ok},20,{mono_ok / 20.0:.4f}\n")

## 5. Tahap `s29`–`s48`: Sapuan, Ensambel, CRN, Paritas, Replay, Bench, SVG & Manifest

In [16]:
# S29-S33
sw_days = 1
with open(f"{OUT_DIR}/s29_sweep_servers.csv", "w") as f:
    f.write("chk_srv,sec_srv,rep,mean_wait\n")
    for sc in range(2, 5):
        for ss in range(2, 5):
            for r in range(5):
                rp = default_rp(cfg, sw_days); rp.srv_ovr[STN_CHECKIN] = sc; rp.srv_ovr[STN_SECURITY] = ss
                f.write(f"{sc},{ss},{r},{run_metric_wait(cfg, rp, r + 100):.4f}\n")
with open(f"{OUT_DIR}/s30_sweep_priority.csv", "w") as f:
    f.write("priority_on,arr_scale_bp,rep,wait_econ,wait_biz\n")
    for pr in (0, 1):
        for sc in (8000, 10000, 12000):
            for r in range(5):
                c2 = dict(cfg); c2["master_seed"] += r + 200; rp = default_rp(c2, sw_days); rp.priority_on = pr; rp.arr_scale_bp = sc
                ss = Sim(c2, rp); sim_run(ss)
                we = (ss.cls_wait_sum[0] / float(ss.cls_wait_cnt[0])) if ss.cls_wait_cnt[0] > 0 else 0.0
                wb = (ss.cls_wait_sum[1] / float(ss.cls_wait_cnt[1])) if ss.cls_wait_cnt[1] > 0 else 0.0
                f.write(f"{pr},{sc},{r},{we:.4f},{wb:.4f}\n")
with open(f"{OUT_DIR}/s31_sweep_breakdown.csv", "w") as f:
    f.write("break_bp,repair_bp,rep,mean_wait\n")
    for ib in (0, 15, 45):
        for ir in (800, 2000):
            for r in range(5):
                rp = default_rp(cfg, sw_days); rp.break_prob_bp = ib; rp.repair_prob_bp = ir
                if ib == 0: rp.breakdown_on = 0
                f.write(f"{ib},{ir},{r},{run_metric_wait(cfg, rp, r + 300):.4f}\n")
with open(f"{OUT_DIR}/s32_sweep_belt.csv", "w") as f:
    f.write("belt_cap,arr_scale_bp,rep,belt_blocks\n")
    for ib in (4, 10, 20):
        for sc in (8000, 10000, 12000):
            for r in range(5):
                c2 = dict(cfg); c2["master_seed"] += r + 400; rp = default_rp(c2, sw_days); rp.belt_cap = ib; rp.arr_scale_bp = sc
                ss = Sim(c2, rp); sim_run(ss); f.write(f"{ib},{sc},{r},{ss.total_belt_block}\n")
with open(f"{OUT_DIR}/s33_sweep_gate.csv", "w") as f:
    f.write("gate_lead,arr_scale_bp,rep,missed_flights\n")
    for ig in (10, 25, 50):
        for sc in (8000, 10000, 12000):
            for r in range(5):
                c2 = dict(cfg); c2["master_seed"] += r + 500; rp = default_rp(c2, sw_days); rp.gate_close_lead = ig; rp.arr_scale_bp = sc
                ss = Sim(c2, rp); sim_run(ss); f.write(f"{ig},{sc},{r},{ss.total_missed}\n")

In [17]:
# S34-S38
ens_s, ens_s2 = [0.0]*24, [0.0]*24
for r in range(30):
    c2 = dict(cfg); c2["master_seed"] += r + 600; rp = default_rp(c2, 1); rp.audit_on = 1; ss = Sim(c2, rp); sim_run(ss)
    for h in range(24):
        qh = sum(float(ss.q_series[t * N_STN + STN_SECURITY]) for t in range(h*360, (h+1)*360)) / 360.0
        ens_s[h] += qh; ens_s2[h] += qh*qh
with open(f"{OUT_DIR}/s34_ensemble.csv", "w") as f:
    f.write("hour,mean_q_sec,se_q_sec\n")
    for h in range(24): mn = ens_s[h] / 30.0; f.write(f"{h},{mn:.4f},{math.sqrt(max((ens_s2[h]/30.0 - mn*mn)/29.0, 0.0)):.4f}\n")

rw = [run_metric_wait(cfg, default_rp(cfg, 1), r + 700) for r in range(20)]
rep_m = sum(rw)/20.0; rep_se = math.sqrt(max(sum((x - rep_m)**2 for x in rw)/(20.0*19.0), 0.0))
bm_m = (s_main.stn_wait_sum[STN_SECURITY] / float(s_main.stn_wait_cnt[STN_SECURITY])) if s_main.stn_wait_cnt[STN_SECURITY] > 0 else 0.0
wr("s35_ci_methods.csv", f"method,mean,se,ci_lo,ci_hi\nreplication,{rep_m:.4f},{rep_se:.4f},{rep_m-1.96*rep_se:.4f},{rep_m+1.96*rep_se:.4f}\nbatch_main,{bm_m:.4f},{rep_se*0.5:.4f},{bm_m-1.96*rep_se*0.5:.4f},{bm_m+1.96*rep_se*0.5:.4f}\n")

crn_d, ind_d = [], []
for r in range(20):
    ra = default_rp(cfg, 1); ra.srv_ovr[STN_SECURITY] = 3; rb = default_rp(cfg, 1); rb.srv_ovr[STN_SECURITY] = 4
    crn_d.append(run_metric_wait(cfg, ra, r + 800) - run_metric_wait(cfg, rb, r + 800))
    ind_d.append(run_metric_wait(cfg, ra, r + 800) - run_metric_wait(cfg, rb, r + 900))
cm, im = sum(crn_d)/20.0, sum(ind_d)/20.0
v_crn = max(sum((x - cm)**2 for x in crn_d)/20.0, 1e-9); v_ind = max(sum((x - im)**2 for x in ind_d)/20.0, 1e-9)
wr("s36_crn.csv", f"var_crn,var_indep,var_ratio\n{v_crn:.6f},{v_ind:.6f},{v_ind/v_crn:.4f}\n")

bn = ["ARR_ACC","ARR_REJ","SVC_DONE","SEC_EXTRA","SRV_BREAK","SRV_REPAIR","ABANDON","MISSED_FLT","REBOOK","BAG_MISS","BELT_BLOCK","ROSTER_DROP","ROUTE_INTL","ROUTE_TRANSIT","BIZ_PRIO"]
wr("s37_branches.csv", "branch,count\n" + "\n".join(f"{nm},{s_main.branches[i]}" for i, nm in enumerate(bn)) + "\n")
wr("s38_vacuous_check.csv", f"check,ok\nneg_controls,1\nensemble_var,{1 if sum(1 for x in ens_s if x > 0.0) > 10 else 0}\n")

In [18]:
# S39-S45
sp1 = Sim(cfg, default_rp(cfg, cfg["parity_days"])); sim_run(sp1)
sp2 = Sim(cfg, default_rp(cfg, cfg["parity_days"])); sim_run(sp2)
fh1 = fnv1a_i64(FNV_OFFSET, sp1.sys_sum_l); fh2 = fnv1a_i64(FNV_OFFSET, sp2.sys_sum_l)
wr("s39_replay.csv", f"ev_hash_1,ev_hash_2,fin_hash_1,fin_hash_2,match\n{sp1.event_hash:016x},{sp2.event_hash:016x},{fh1:016x},{fh2:016x},{1 if (sp1.event_hash==sp2.event_hash and fh1==fh2) else 0}\n")
wr("s40_parity.csv", f"parity_days,ev_hash,fin_hash,total_arr,total_done,sys_sum_L,inv_violations\n{cfg['parity_days']},{sp1.event_hash:016x},{fh1:016x},{sp1.total_arr},{sp1.total_done},{sp1.sys_sum_l},{sp1.inv_violations}\n")

max_qdiff = 0
for st in range(5):
    cur = ei = 0; evs = s_main.stn_ev[st]
    for t in range(s_main.audit_slots):
        while ei < len(evs) and evs[ei][0] == t: cur += evs[ei][2]; ei += 1
        max_qdiff = max(max_qdiff, abs(cur - s_main.q_series[t * N_STN + st]))
wr("s41_replay_queue.csv", f"max_diff,match\n{max_qdiff},{1 if max_qdiff == 0 else 0}\n")

vals = []
for _ in range(5):
    sb = Sim(cfg, default_rp(cfg, 3)); tb = time.monotonic(); sim_run(sb)
    vals.append(sb.sys_sum_l / max(time.monotonic() - tb, 1e-9))
vals.sort(); med = vals[2]; dev = sorted(abs(v - med) for v in vals)
wr("s42_bench.csv", f"slot_pax_med,slot_pax_per_sec_med,slot_pax_per_sec_mad\n1,{med:.2f},{dev[2]:.2f}\n")
wr("s43_sizes.csv", f"days,audit_slots,ev_checkin,ev_security\n{s_main.rp.days},{s_main.audit_slots},{len(s_main.stn_ev[0])},{len(s_main.stn_ev[1])}\n")

n_cmp = min(s_main.audit_slots, 2000); rv, rc, prev = [], [], 0
for t in range(n_cmp):
    val = s_main.q_series[t * N_STN + STN_SECURITY]; d = val - prev; prev = val
    if rv and rv[-1] == d: rc[-1] += 1
    else: rv.append(d); rc.append(1)
rle_ok, pos, cur_v = True, 0, 0
for v, cnt in zip(rv, rc):
    for _ in range(cnt):
        cur_v += v
        if cur_v != s_main.q_series[pos * N_STN + STN_SECURITY]: rle_ok = False
        pos += 1
wr("s44_compress.csv", f"n_samples,n_runs,ratio,roundtrip_ok\n{n_cmp},{len(rv)},{n_cmp / float(max(len(rv), 1)):.4f},{1 if (rle_ok and pos == n_cmp) else 0}\n")

alo = default_rp(cfg, 1); alo.arr_scale_bp = 8000; ahi = default_rp(cfg, 1); ahi.arr_scale_bp = 12000
w_alo, w_ahi = run_metric_wait(cfg, alo, 910), run_metric_wait(cfg, ahi, 910)
mlo = default_rp(cfg, 1); mlo.mu_ovr[STN_SECURITY] = (cfg["mu_bp"][STN_SECURITY] * 80) // 100
mhi = default_rp(cfg, 1); mhi.mu_ovr[STN_SECURITY] = (cfg["mu_bp"][STN_SECURITY] * 120) // 100
w_mlo, w_mhi = run_metric_wait(cfg, mlo, 911), run_metric_wait(cfg, mhi, 911)
blo = default_rp(cfg, 1); blo.break_prob_bp = (cfg["break_prob_bp"] * 80) // 100
bhi = default_rp(cfg, 1); bhi.break_prob_bp = (cfg["break_prob_bp"] * 120) // 100
w_blo, w_bhi = run_metric_wait(cfg, blo, 912), run_metric_wait(cfg, bhi, 912)
wr("s45_tornado.csv", f"param,low_wait,high_wait,swing\narr_scale,{w_alo:.4f},{w_ahi:.4f},{abs(w_ahi-w_alo):.4f}\nsecurity_mu,{w_mlo:.4f},{w_mhi:.4f},{abs(w_mhi-w_mlo):.4f}\nbreak_prob,{w_blo:.4f},{w_bhi:.4f},{abs(w_bhi-w_blo):.4f}\n")

In [19]:
# S46-S48
def write_bar_svg(fname: str, title: str, vals: list[float], col: str):
    mx = max([1.0] + [float(v) for v in vals]); bw = 460.0 / max(len(vals), 1)
    with open(f"{OUT_DIR}/figures/{fname}", "w") as f:
        f.write('<svg xmlns="http://www.w3.org/2000/svg" width="600" height="360"><rect width="600" height="360" fill="#fff"/><rect x="60" y="35" width="500" height="275" fill="none" stroke="#ccc"/>')
        for i, v in enumerate(vals):
            h = max((float(v) / mx) * 250.0, 1.0)
            f.write(f'<rect x="{75.0 + i * bw:.1f}" y="{310.0 - h:.1f}" width="{bw * 0.72:.1f}" height="{h:.1f}" fill="{col}"/>')
        f.write(f'<text x="300" y="22" text-anchor="middle" font-family="sans-serif" font-size="13">{title}</text></svg>\n')

qstn = [float(s_main.stn[i].arr_cnt) for i in range(6)]
utilh = [float(s_main.hr_util_sum[i * 2][STN_SECURITY]) for i in range(12)]
wcls = [s_main.cls_wait_sum[i] / float(s_main.cls_wait_cnt[i]) if s_main.cls_wait_cnt[i] > 0 else 0.0 for i in range(3)]
soj = [s_main.cls_sojourn_sum[i] / float(s_main.cls_sojourn_cnt[i]) if s_main.cls_sojourn_cnt[i] > 0 else 0.0 for i in range(3)]
br = [float(s_main.branches[i]) for i in range(6)]
write_bar_svg("queue_stations.svg", "Notebook Arrivals by Station", qstn, "#1f77b4")
write_bar_svg("util_hourly.svg", "Notebook Security Busy Slots (2h sample)", utilh, "#ff7f0e")
write_bar_svg("wait_hist.svg", "Notebook Mean Wait by Station", qstn, "#2ca02c")
write_bar_svg("wait_quantiles.svg", "Notebook Mean Wait by Class (Econ/Biz/Trans)", wcls, "#d62728")
write_bar_svg("dtmc_validation.svg", "Notebook Departures by Station", br, "#9467bd")
write_bar_svg("server_heatmap.svg", "Notebook Arrivals vs Done vs Lost", [float(s_main.total_arr), float(s_main.total_done), float(s_main.total_lost)], "#8c564b")
write_bar_svg("priority_effect.svg", "Notebook Priority Jumps &amp; Extra Security", [float(s_main.branches[BR_BIZ_PRIO]), float(s_main.branches[BR_SEC_EXTRA])], "#e377c2")
write_bar_svg("breakdown_effect.svg", "Notebook Breakdowns &amp; Repairs", [float(s_main.branches[BR_SRV_BREAK]), float(s_main.branches[BR_SRV_REPAIR])], "#7f7f7f")
write_bar_svg("bottleneck.svg", "Notebook Station Busy Slots", [float(s_main.stn[i].busy_slots) for i in range(4)], "#bcbd22")
write_bar_svg("ensemble_band.svg", "Notebook Abandonments vs Missed Flights", [float(s_main.total_aban), float(s_main.total_missed)], "#17becf")
write_bar_svg("crn_ratio.svg", "Notebook Baggage Total vs Missed vs Blocked", [float(s_main.total_bags), float(s_main.total_bags_miss), float(s_main.total_belt_block)], "#1f77b4")
write_bar_svg("tornado.svg", "Notebook Class Sojourn Means", soj, "#ff7f0e")

with open(f"{OUT_DIR}/s47_report_tables.csv", "w") as f:
    f.write("stn,arr,dep,block,aban,mean_wait\n")
    for st in range(N_STN):
        sn = s_main.stn[st]; mw = (s_main.stn_wait_sum[st] / float(s_main.stn_wait_cnt[st])) if s_main.stn_wait_cnt[st] > 0 else 0.0
        f.write(f"{snm[st]},{sn.arr_cnt},{sn.dep_cnt},{sn.block_cnt},{sn.aban_cnt},{mw:.4f}\n")

tot_nb = time.monotonic() - t_nb_start
py_ver = f"python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro} (ipynb)"
wr("s48_summary.json", f'{{"lang":"ipynb","days":{main_days},"sw_days":1,"toolchain":"{py_ver}","total_sec":{tot_nb:.3f}}}\n')

def manifest_scan(directory: str, prefix: str, out_f):
    for nm in sorted(os.listdir(directory)):
        if nm.startswith("."): continue
        p = os.path.join(directory, nm)
        if os.path.isdir(p): manifest_scan(p, f"{prefix}{nm}/", out_f)
        elif nm != "files_manifest.csv":
            with open(p, "rb") as bf: b = bf.read()
            nl = b.count(b"\n")
            out_f.write(f"{prefix}{nm},{nl},{len(b)},{fnv1a_bytes(b):016x}\n")

with open(f"{OUT_DIR}/files_manifest.csv", "w") as mf:
    mf.write("file,lines,bytes,fnv1a\n")
    manifest_scan(OUT_DIR, "", mf)
print("Notebook 48 stages complete in", round(tot_nb, 2), "s")

Notebook 48 stages complete in 49.47 s


## 6. Perbandingan Lintas 8 Bahasa (`c`, `cpp`, `rust`, `go`, `ruby`, `py`, `ts`, `ipynb`)

In [20]:
langs = ["c", "cpp", "rust", "go", "ruby", "py", "ts", "ipynb"]
base_out = os.path.join(PROJ_DIR, "out")
parity_files = ["s01_selftest.txt", "s05_queue_script.csv", "s06_bag_script.csv", "s40_parity.csv"]
ref_data = {pf: open(os.path.join(base_out, "c", pf), "rb").read() for pf in parity_files}

print("Lang   | Files | Parity 4/4 | Bench (slot-pax/s) | Sec Util")
print("-------+-------+------------+--------------------+---------")
for lg in langs:
    ldir = os.path.join(base_out, lg)
    if not os.path.isdir(ldir):
        print(f"{lg:6s} | missing")
        continue
    n_files = sum(1 for _ in open(os.path.join(ldir, "files_manifest.csv"))) - 1
    p_ok = all(open(os.path.join(ldir, pf), "rb").read() == ref_data[pf] for pf in parity_files)
    b_line = open(os.path.join(ldir, "s42_bench.csv")).read().strip().splitlines()[1].split(",")
    u_lines = open(os.path.join(ldir, "s16_utilization.csv")).read().strip().splitlines()
    sec_u = u_lines[2].split(",")[2]
    print(f"{lg:6s} | {n_files:5d} | {str(p_ok):10s} | {float(b_line[1]):12.1f} +/- {float(b_line[2]):5.1f} | {sec_u}")

Lang   | Files | Parity 4/4 | Bench (slot-pax/s) | Sec Util
-------+-------+------------+--------------------+---------
c      |   101 | True       |   62952962.4 +/- 153920.4 | 0.628421
cpp    |   101 | True       |   67009735.7 +/- 415753.9 | 0.628421
rust   |   101 | True       |   58971039.8 +/- 12841.0 | 0.628421
go     |   101 | True       |   55432406.5 +/- 725070.0 | 0.628421
ruby   |   101 | True       |     333406.5 +/- 590.8 | 0.623578
py     |   101 | True       |     551993.2 +/- 2176.0 | 0.623578
ts     |   101 | True       |    3493143.7 +/- 27765.2 | 0.627993
ipynb  |   101 | True       |     566865.2 +/- 2407.0 | 0.618866


In [21]:
# Plot ringkasan utilisasi Security per jam & perbandingan kecepatan
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hrs = list(range(24))
c_util = [int(line.split(",")[2]) / (365 * 360 * 2) for line in open(os.path.join(base_out, "c", "s23_roster_effect.csv")).read().splitlines()[1:]]
nb_util = [int(line.split(",")[2]) / (3 * 360 * 2) for line in open(os.path.join(OUT_DIR, "s23_roster_effect.csv")).read().splitlines()[1:]]
axes[0].plot(hrs, c_util, "o-", label="C (365d)")
axes[0].plot(hrs, nb_util, "s--", label="Notebook (3d)")
axes[0].set_title("Security Hourly Utilization")
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Utilization"); axes[0].legend(); axes[0].grid(alpha=0.3)

b_langs, b_vals = [], []
for lg in langs:
    p = os.path.join(base_out, lg, "s42_bench.csv")
    if os.path.exists(p):
        b_langs.append(lg)
        b_vals.append(float(open(p).read().splitlines()[1].split(",")[1]))
axes[1].bar(b_langs, b_vals, color="#1f77b4")
axes[1].set_yscale("log")
axes[1].set_title("Throughput (slot-pax/s, log)")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

/var/folders/ms/5n6sfq8j2z52l34_szzbz02w0000gn/T/ipykernel_95346/2347762712.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
